# 01 — Dataset Analysis

**Facial Skin Condition Detection — MCA Project**

This notebook:
- Scans the `dataset/` directory
- Validates all images (readability, format)
- Counts images per class
- Detects class imbalance
- Detects duplicate images
- Generates visual charts
- Saves a dataset summary CSV

---
> ⚠️ **Before running**: Place your images in `dataset/Acne/`, `dataset/Rosacea/`, etc.

In [ ]:
# ── Setup: add project root to path so we can import src/ modules ──────────
import sys
import os
from pathlib import Path

# Navigate two levels up from notebooks/ to reach the project root
PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset dir : {PROJECT_ROOT / 'dataset'}")

In [ ]:
# ── Import the dataset analysis module ─────────────────────────────────────
from src.dataset_analysis import (
    scan_dataset,
    print_summary,
    save_summary_csv,
    plot_class_distribution,
    print_recommendations,
    CLASS_FOLDERS,
    DATASET_DIR,
)

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('inline')     # Jupyter inline display
%matplotlib inline

print("Imports successful.")

## Step 1 — Scan the Dataset

In [ ]:
# Run the full dataset scan
results = scan_dataset()

## Step 2 — Print the Summary Report

In [ ]:
print_summary(results)

## Step 3 — Class Distribution Table

In [ ]:
# Build a neat DataFrame for display
rows = []
for class_name, stats in results['class_stats'].items():
    rows.append({
        'Class':        class_name,
        'Folder Found': 'YES' if stats['folder_found'] else 'NO',
        'Total Files':  stats['total_files'],
        'Valid':        stats['valid'],
        'Invalid':      stats['invalid'],
    })

df_summary = pd.DataFrame(rows)
display(df_summary)

## Step 4 — Visual Distribution Chart

In [ ]:
import seaborn as sns
import numpy as np

palette = ['#E74C3C', '#E67E22', '#F1C40F', '#2ECC71', '#3498DB']

classes = list(results['class_stats'].keys())
counts  = [results['class_stats'][c]['valid'] for c in classes]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Dataset Class Distribution', fontsize=16, fontweight='bold')

# Bar chart
bars = axes[0].bar(classes, counts, color=palette, edgecolor='black')
axes[0].set_title('Valid Images per Class')
axes[0].set_xlabel('Skin Condition')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=20)
for bar, cnt in zip(bars, counts):
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.3,
        str(cnt), ha='center', va='bottom', fontweight='bold'
    )

# Pie chart
non_zero = [(c, n) for c, n in zip(classes, counts) if n > 0]
if non_zero:
    pie_labels, pie_counts = zip(*non_zero)
    axes[1].pie(
        pie_counts, labels=pie_labels,
        autopct='%1.1f%%', colors=palette[:len(non_zero)],
        startangle=140,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
    )
    axes[1].set_title('Class Proportion')
else:
    axes[1].text(0.5, 0.5, 'No data yet — add images to dataset/', 
                 ha='center', va='center', fontsize=12)
    axes[1].set_title('Class Proportion')

plt.tight_layout()
plt.show()

## Step 5 — Per-Image Detail Table

In [ ]:
# Flatten all image records into one DataFrame
all_records = []
for class_name, stats in results['class_stats'].items():
    all_records.extend(stats['image_records'])

if all_records:
    df_images = pd.DataFrame(all_records)
    print(f"Total image records: {len(df_images)}")
    display(df_images.head(20))
else:
    print("No image records found. Add images to the dataset/ folders first.")

## Step 6 — Image Dimension Analysis

In [ ]:
sizes = results['image_sizes']

if sizes:
    heights = [s[0] for s in sizes]
    widths  = [s[1] for s in sizes]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Image Dimension Distribution', fontsize=14, fontweight='bold')

    axes[0].hist(heights, bins=20, color='#3498DB', edgecolor='black', alpha=0.8)
    axes[0].set_title('Image Heights')
    axes[0].set_xlabel('Height (px)')
    axes[0].set_ylabel('Count')
    axes[0].axvline(np.mean(heights), color='red', linestyle='--', label=f'Mean: {np.mean(heights):.0f}px')
    axes[0].legend()

    axes[1].hist(widths, bins=20, color='#2ECC71', edgecolor='black', alpha=0.8)
    axes[1].set_title('Image Widths')
    axes[1].set_xlabel('Width (px)')
    axes[1].set_ylabel('Count')
    axes[1].axvline(np.mean(widths), color='red', linestyle='--', label=f'Mean: {np.mean(widths):.0f}px')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print(f"\nHeight — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")
    print(f"Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
else:
    print("No images found. Cannot plot dimensions.")

## Step 7 — Save Summary CSV

In [ ]:
save_summary_csv(results)
print("Dataset summary saved to features/dataset_summary.csv")

## Step 8 — Recommendations

In [ ]:
print_recommendations(results)

---
## ✅ Phase 1 Complete

Next steps:
1. Add images to the dataset/ folders if not already done
2. Proceed to **02_preprocessing.ipynb** for image preprocessing

---
*This is an academic MCA project. Results are not medical diagnoses.*